# 常量和设置

In [1]:
import random
import numpy as np
import pandas as pd
import os
import sys
import warnings
from pathlib import Path
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
sys.path.append(str(PROJECT_ROOT))
from src.constants.train_constants import *
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)

DATA_PATH = Path("/home_ext/zzx/work_file/ATEL-Correction-Analysis-of-Recipe/raw_data/df_filtered_outlier.csv")
MODEL_DIR = Path("/home_ext/zzx/work_file/ATEL-Correction-Analysis-of-Recipe/models_result")
RESULT_DIR = Path("/home_ext/zzx/work_file/ATEL-Correction-Analysis-of-Recipe/results")

warnings.filterwarnings("ignore")
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)


In [2]:
from datetime import datetime
# 当前日期，格式为 YYYYMMDD
run_date = datetime.now().strftime("%Y%m%d")
print(run_date)

20260209


In [2]:
df = pd.read_csv(DATA_PATH)
df.columns

Index(['index', 'batch_log', 'machine_sn', 'hc_chamber_day', 'batch_number',
       'is_centralized', 'set_rate_nm_sec_1', 'rate_coefficient_1',
       'act_rate_nm_sec_1', 'set_rate_nm_sec_2', 'rate_coefficient_2',
       'act_rate_nm_sec_2', 'set_rate_nm_sec_3', 'rate_coefficient_3',
       'act_rate_nm_sec_3', 'set_rate_nm_sec_4', 'rate_coefficient_4',
       'act_rate_nm_sec_4', 'set_rate_nm_sec_5', 'rate_coefficient_5',
       'act_rate_nm_sec_5', 'set_rate_nm_sec_6', 'rate_coefficient_6',
       'act_rate_nm_sec_6', 'set_rate_nm_sec_7', 'rate_coefficient_7',
       'act_rate_nm_sec_7', 'set_rate_nm_sec_8', 'rate_coefficient_8',
       'act_rate_nm_sec_8', 'set_rate_nm_sec_9', 'rate_coefficient_9',
       'act_rate_nm_sec_9', 'set_rate_nm_sec_10', 'rate_coefficient_10',
       'act_rate_nm_sec_10', 'set_rate_nm_sec_11', 'rate_coefficient_11',
       'act_rate_nm_sec_11', 'c3_10_y_min', 'c3_10_y_max', 'c3_10_y_mean',
       'c3_10_a_min', 'c3_10_a_max', 'c3_10_a_mean', 'c3_10_b_min

In [4]:
# 树模型设置
from src.constants.data_constants import *
# 分类特征
TREE_CAT_COLS = (
    MACHINE_COLS
    # + DATE_FEATURE_COLS[1:2]
)
print(TREE_CAT_COLS)
# 数值特征
TREE_NUM_COLS = (
    ACT_RATE_NM_SEC_COLS
#    + RATE_NM_SEC_COLS
    + RATE_COEF_COLS
    + BATCH_NUMBER_COLS
)
print(TREE_NUM_COLS)
# 预测变量
TREE_TARGET_COLS = COLOR_COLS + TRANSMITTANCE_COLS
print(TREE_TARGET_COLS)

['machine_sn']
['act_rate_nm_sec_1', 'act_rate_nm_sec_2', 'act_rate_nm_sec_3', 'act_rate_nm_sec_4', 'act_rate_nm_sec_5', 'act_rate_nm_sec_6', 'act_rate_nm_sec_7', 'act_rate_nm_sec_8', 'act_rate_nm_sec_9', 'act_rate_nm_sec_10', 'act_rate_nm_sec_11', 'rate_coefficient_1', 'rate_coefficient_2', 'rate_coefficient_3', 'rate_coefficient_4', 'rate_coefficient_5', 'rate_coefficient_6', 'rate_coefficient_7', 'rate_coefficient_8', 'rate_coefficient_9', 'rate_coefficient_10', 'rate_coefficient_11', 'batch_number']
['c3_10_y_min', 'c3_10_y_max', 'c3_10_a_min', 'c3_10_a_max', 'c3_10_b_min', 'c3_10_b_max', 'c3_45_a_min', 'c3_45_a_max', 'c3_45_b_min', 'c3_45_b_max', 't_0deg_400_770_avg_min', 't_0deg_940_min', 't_40deg_920_960_avg_min']


# 特征工程和数据集预处理

In [5]:
from src.data.preprocess_and_split import preprocess_and_split
data, tree_train, tree_valid = preprocess_and_split(
    raw_data_path=DATA_PATH,
    num_cols=TREE_NUM_COLS,
    cat_cols=TREE_CAT_COLS,
    target_cols=TREE_TARGET_COLS,
    other_info_cols=OTHER_INFO_COLS,
    raw_date_cols=RAW_DATE_COLS,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)
for col in data.columns:
    print(col, len(data[col].unique()))

index 21472
batch_log 21472
hc_chamber_day 213
act_rate_nm_sec_1 1
act_rate_nm_sec_2 1
act_rate_nm_sec_3 40
act_rate_nm_sec_4 1111
act_rate_nm_sec_5 64
act_rate_nm_sec_6 94
act_rate_nm_sec_7 432
act_rate_nm_sec_8 833
act_rate_nm_sec_9 352
act_rate_nm_sec_10 522
act_rate_nm_sec_11 552
rate_coefficient_1 1
rate_coefficient_2 1
rate_coefficient_3 13
rate_coefficient_4 279
rate_coefficient_5 40
rate_coefficient_6 70
rate_coefficient_7 270
rate_coefficient_8 573
rate_coefficient_9 221
rate_coefficient_10 351
rate_coefficient_11 296
batch_number 307
machine_sn 41
c3_10_y_min 16096
c3_10_y_max 16836
c3_10_a_min 21099
c3_10_a_max 21200
c3_10_b_min 20273
c3_10_b_max 20038
c3_45_a_min 21189
c3_45_a_max 21174
c3_45_b_min 19483
c3_45_b_max 20949
t_0deg_400_770_avg_min 17384
t_0deg_940_min 2576
t_40deg_920_960_avg_min 18330


In [6]:
print(f"训练集大小: {len(tree_train)}, 验证集大小: {len(tree_valid)}")
print(tree_train.head(1))

训练集大小: 17177, 验证集大小: 4295
       index               batch_log hc_chamber_day  act_rate_nm_sec_1  \
18338  18367  A07HB03U3037-250904093     2025-09-04                1.0   

       act_rate_nm_sec_2  act_rate_nm_sec_3  act_rate_nm_sec_4  \
18338                1.0             0.8128           0.626415   

       act_rate_nm_sec_5  act_rate_nm_sec_6  act_rate_nm_sec_7  \
18338             0.4555            1.03887           0.422021   

       act_rate_nm_sec_8  act_rate_nm_sec_9  act_rate_nm_sec_10  \
18338           0.895132           0.452311            0.923044   

       act_rate_nm_sec_11  rate_coefficient_1  rate_coefficient_2  \
18338            0.382709                 1.0                 1.0   

       rate_coefficient_3  rate_coefficient_4  rate_coefficient_5  \
18338                 1.0               0.997                 1.0   

       rate_coefficient_6  rate_coefficient_7  rate_coefficient_8  \
18338              1.0198              0.9265              0.8787   

       

In [ ]:
from src.analysis.data_description import batch_plot_targets
desc_path = RESULT_DIR / f"target_analysis/description/desc_{run_date}"
is_desc = False
if is_desc:
    batch_plot_targets(
        df=data,
        feature_cols=TREE_CAT_COLS + TREE_NUM_COLS,
        target_cols=TREE_TARGET_COLS,
        output_dir=desc_path,
    )

# 训练模型和结果输出

# XGBoost

In [7]:
all_xgb_path= MODEL_DIR / f"xgb/ipynb_test/xgb2_{run_date}"
load_xgb_path= MODEL_DIR / f"xgb/xgb_20260128"
xgb_load = False

In [ ]:
# XGB 读取或训练
from src.models.xgb_regressor import XGBRegressor
from src.constants.train_constants import *
all_xgb = {}
for target in TREE_TARGET_COLS:
    if not xgb_load:
        print(f"Training XGBoost for {target}")
        all_xgb[target] = XGBRegressor(
            num_cols=TREE_NUM_COLS,
            cat_cols=TREE_CAT_COLS,
            n_trials=TREE_DEFAULT_N_TRIALS,
            n_jobs=6,
            random_state=RANDOM_STATE
        )
        all_xgb[target].fit(
            tree_train,
            tree_valid,
            [target]
        )
        all_xgb[target].save_model(
            dir_path = all_xgb_path
        )
    else:
        print(f"Loading XGBoost for {target}")
        all_xgb[target] = XGBRegressor.load_model(
            model_dir= load_xgb_path/target
        )


Training XGBoost for c3_10_y_min


  0%|          | 0/100 [00:00<?, ?it/s]

[0]	valid-rmse:0.09122
[400]	valid-rmse:0.06881
[460]	valid-rmse:0.06881
Training XGBoost for c3_10_y_max


  0%|          | 0/100 [00:00<?, ?it/s]

[0]	valid-rmse:0.09959
[55]	valid-rmse:0.08169
Training XGBoost for c3_10_a_min


...

[0]	valid-rmse:0.17850
[53]	valid-rmse:0.17117
Training XGBoost for t_0deg_940_min


  0%|          | 0/100 [00:00<?, ?it/s]

[0]	valid-rmse:0.26506
[54]	valid-rmse:0.23509
Training XGBoost for t_40deg_920_960_avg_min


  0%|          | 0/100 [00:00<?, ?it/s]

[0]	valid-rmse:0.63460
[60]	valid-rmse:0.55603


In [13]:
from src.analysis.anomaly_mining import export_prediction_anomalies
xgb_anomaly_dir = RESULT_DIR / f"anomaly_records/xgb_abs_anomaly_top100_{run_date}"
check_anomaly = False
if check_anomaly:
    export_prediction_anomalies(
        models=all_xgb,
        train=tree_train,
        valid=tree_valid,
        feature_cols=TREE_FEATURE_COLS,
        target_cols=TREE_TARGET_COLS,
        output_dir=xgb_anomaly_dir,
        error_type="abs",
        top_k=100
    )
    tree_data.to_csv(xgb_anomaly_dir / "all_data.csv", index=False)

In [9]:
from src.evaluation.evaluator import RegressionEvaluator
from src.constants.data_constants import *
xgb_eval_path = RESULT_DIR / f"xgb_eval/ipynb_test/xgb2_eval_{run_date}"
# xgb_eval_path = RESULT_DIR / f"xgb_eval/xgb_eval_20260107"
# xgb_eval_path = RESULT_DIR / f"xgb_eval/eval_20251230_date_test"
xgb_evaluator = RegressionEvaluator(
    num_cols=TREE_NUM_COLS,
    cat_cols=TREE_CAT_COLS,
    output_dir=xgb_eval_path
)
xgb_evaluator.run_full_evaluation(
    regressors=all_xgb,
    train=tree_train,
    valid=tree_valid,
    target_cols=TREE_TARGET_COLS
)

  0%|          | 0/1500 [00:00<?, ?it/s]

  0%|          | 0/1500 [00:00<?, ?it/s]

  0%|          | 0/1500 [00:00<?, ?it/s]

  0%|          | 0/1500 [00:00<?, ?it/s]

  0%|          | 0/1500 [00:00<?, ?it/s]

...

  0%|          | 0/1500 [00:00<?, ?it/s]

  0%|          | 0/1500 [00:00<?, ?it/s]

  0%|          | 0/1500 [00:00<?, ?it/s]

  0%|          | 0/1500 [00:00<?, ?it/s]

  0%|          | 0/1500 [00:00<?, ?it/s]

# LGBM

In [11]:
all_lgbm_path = MODEL_DIR / f"lgbm/lgbm_{run_date}"
load_lgbm_path = MODEL_DIR / f"lgbm/lgbm_20260128"
lgbm_load = True

In [ ]:
# LGBM 训练或加载
from src.models.lgbm_regressor import LGBMRegressor
all_lgbm = {}
for target in TREE_TARGET_COLS:
    if not lgbm_load:
        print(f"Training LGBM for {target}")
        all_lgbm[target] = LGBMRegressor(
            num_cols=TREE_NUM_COLS,
            cat_cols=TREE_CAT_COLS,
            n_trials=N_TRIALS,
            n_jobs=1,
            random_state=RANDOM_STATE
        )
        all_lgbm[target].fit(
            tree_train,
            tree_valid,
            [target]
        )
        all_lgbm[target].save_model(
            path = all_lgbm_path / target
        )
    else:
        print(f"Loading LGBM for {target}")
        all_lgbm[target] = LGBMRegressor.load_model(
            dir_path=load_lgbm_path / target
        )

Loading LGBM for c3_10_y_min
Loading LGBM for c3_10_y_max
Loading LGBM for c3_10_a_min
Loading LGBM for c3_10_a_max
Loading LGBM for c3_10_b_min
Loading LGBM for c3_10_b_max
Loading LGBM for c3_45_a_min
Loading LGBM for c3_45_a_max
Loading LGBM for c3_45_b_min
Loading LGBM for c3_45_b_max
Loading LGBM for t_0deg_400_770_avg_min
Loading LGBM for t_0deg_940_min
Loading LGBM for t_40deg_920_960_avg_min


In [13]:
from src.analysis.anomaly_mining import export_prediction_anomalies
lgbm_anomaly_dir = RESULT_DIR / f"anomaly_records/lgbm_abs_anomaly_top100_{run_date}"
lgbm_anomaly_dir = RESULT_DIR / f"anomaly_records/lgbm_abs_anomaly_top100_20260128"
check_anomaly = True
if check_anomaly:
    export_prediction_anomalies(
        models=all_lgbm,
        train=tree_train,
        valid=tree_valid,
        feature_cols=TREE_FEATURE_COLS,
        target_cols=TREE_TARGET_COLS,
        output_dir=lgbm_anomaly_dir,
        error_type="abs",
        top_k=100
    )
    tree_data.to_csv(lgbm_anomaly_dir / "all_data.csv", index=False)

目标变量 c3_10_y_min 异常样本 Top 100 已保至 /home_ext/zzx/work_file/ATEL-Correction-Analysis-of-Recipe/results/anomaly_records/lgbm_abs_anomaly_top100_20260128/c3_10_y_min_anomalies.csv
目标变量 c3_10_y_max 异常样本 Top 100 已保至 /home_ext/zzx/work_file/ATEL-Correction-Analysis-of-Recipe/results/anomaly_records/lgbm_abs_anomaly_top100_20260128/c3_10_y_max_anomalies.csv
目标变量 c3_10_a_min 异常样本 Top 100 已保至 /home_ext/zzx/work_file/ATEL-Correction-Analysis-of-Recipe/results/anomaly_records/lgbm_abs_anomaly_top100_20260128/c3_10_a_min_anomalies.csv
目标变量 c3_10_a_max 异常样本 Top 100 已保至 /home_ext/zzx/work_file/ATEL-Correction-Analysis-of-Recipe/results/anomaly_records/lgbm_abs_anomaly_top100_20260128/c3_10_a_max_anomalies.csv
目标变量 c3_10_b_min 异常样本 Top 100 已保至 /home_ext/zzx/work_file/ATEL-Correction-Analysis-of-Recipe/results/anomaly_records/lgbm_abs_anomaly_top100_20260128/c3_10_b_min_anomalies.csv
目标变量 c3_10_b_max 异常样本 Top 100 已保至 /home_ext/zzx/work_file/ATEL-Correction-Analysis-of-Recipe/results/anomaly_records/lgb

In [14]:
anomaly_test_record = tree_data[tree_data["index"] == 17294]
target = "c3_10_a_min"
real = anomaly_test_record[target]
pred = all_lgbm[target].predict(anomaly_test_record)
print(real, pred)

17293    3.22503
Name: c3_10_a_min, dtype: float64 [-0.51264009]


In [13]:
from src.evaluation.evaluator import RegressionEvaluator
from src.constants import *
lgbm_eval_path = RESULT_DIR / f"lgbm_eval/lgbm_eval_{run_date}"
# lgbm_eval_path = RESULT_DIR / f"lgbm_eval/lgbm_eval_{run_date}"
lgbm_evaluator = RegressionEvaluator(
    num_cols=TREE_NUM_COLS,
    cat_cols=TREE_CAT_COLS,
    output_dir=lgbm_eval_path
)
lgbm_evaluator.run_full_evaluation(
    regressors=all_lgbm,
    train=tree_train,
    valid=tree_valid,
    target_cols=TREE_TARGET_COLS
)

100%|██████████| 1536/1536 [04:31<00:00,  5.66it/s]


# CatBoost

# transfomer-base model

In [2]:
# 基础设置
import torch
import omegaconf
import typing
import collections
import os

torch.serialization.add_safe_globals([
    dict,
    list,
    int,
    float,
    str,
    collections.defaultdict,
    omegaconf.dictconfig.DictConfig,
    omegaconf.listconfig.ListConfig,
    omegaconf.nodes.AnyNode,
    omegaconf.base.ContainerMetadata,
    omegaconf.base.Metadata,
    typing.Any
])

os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [3]:
from src.models.gandalf_regressor import GandalfRegressor
gandalf_model_dir = Path("/home_ext/zzx/work_file/ATEL-Correction-Analysis-of-Recipe/models_result/gandalf/gandalf_20260212_172607")
color_model_dir =gandalf_model_dir / "color"
color_gandalf_model = GandalfRegressor.load_model(
    save_dir=color_model_dir / "model/best_model"
)
print(color_gandalf_model.date_feature_names)
trans_model_dir = gandalf_model_dir / "trans"
trans_gandalf_model = GandalfRegressor.load_model(
    save_dir=trans_model_dir / "model/best_model"
)
print(trans_gandalf_model.date_feature_names)

2026-02-24 11:18:41,926 - {pytorch_tabular.tabular_model:171} - INFO - Experiment Tracking is turned off

2026-02-24 11:18:41,935 - {pytorch_tabular.tabular_model:342} - INFO - Preparing the Trainer

Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.model_summary.ModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


['hc_chamber_day']


2026-02-24 11:18:42,161 - {pytorch_tabular.tabular_model:171} - INFO - Experiment Tracking is turned off

2026-02-24 11:18:42,164 - {pytorch_tabular.tabular_model:342} - INFO - Preparing the Trainer

Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.model_summary.ModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


['hc_chamber_day']


In [4]:
print(color_gandalf_model.continuous_cols == trans_gandalf_model.continuous_cols)
print(color_gandalf_model.categorical_cols == trans_gandalf_model.categorical_cols)

True
True


In [6]:
from src.data.preprocess_and_split import preprocess_and_split
from src.constants.data_constants import TARGET_GROUPS, OTHER_INFO_COLS
all_targets = [x for v in TARGET_GROUPS.values() for x in v]
df, train, valid = preprocess_and_split(
    raw_data_path=DATA_PATH,
    num_cols=color_gandalf_model.continuous_cols,
    cat_cols=color_gandalf_model.categorical_cols,
    target_cols=all_targets,
    other_info_cols=OTHER_INFO_COLS,
    raw_date_cols=color_gandalf_model.date_feature_names,
    test_size=0.2,
    random_state=42,
)
print(len(train))
print(len(valid))

35292
8824


In [7]:
from src.evaluation.evaluator import RegressionEvaluator
gandalf_models = [color_gandalf_model, trans_gandalf_model]
eval_dirs = [color_model_dir / "eval", trans_model_dir / "eval"]
for model, eval_dir in zip(gandalf_models, eval_dirs):
    eval_dir.mkdir(parents=True, exist_ok=True)
    evaluator = RegressionEvaluator(
        num_cols=model.continuous_cols,
        cat_cols=model.categorical_cols,
        date_cols=model.date_feature_names,
        output_dir=eval_dir
    )
    evaluator.run_full_evaluation(
        regressors=model,
        train=train,
        valid=valid,
        target_cols=model.target_cols
    )


  0%|          | 0/2941 [00:00<?, ?it/s]

  0%|          | 0/2941 [00:00<?, ?it/s]

# GANDALF

In [11]:
from src.models.gandalf_regressor import GandalfRegressor
gandalf_color_model_dir = MODEL_DIR / f"gandalf/gandalf_{run_date}/color_no_std"
gandalf_transmittance_model_dir = MODEL_DIR / f"gandalf/gandalf_{run_date}/transmittance_no_std"
gandalf_model_paths = {
    "color": gandalf_color_model_dir,
    "transmittance": gandalf_transmittance_model_dir
}
gandalf_model_load_paths = {
    "color": MODEL_DIR / f"gandalf/gandalf_20260112/color_no_std/best_model",
    "transmittance": MODEL_DIR / f"gandalf/gandalf_20260112/transmittance_no_std/best_model"
}
all_gandalf = {}
gandalf_load = False
if not gandalf_load:
    for (tgt_name, model_path), tgts in zip(gandalf_model_paths.items(), TB_TARGETS_LIST):
        all_gandalf[tgt_name] = GandalfRegressor(
            categorical_cols=TB_CAT_COLS,
            continuous_cols=TB_NUM_COLS,
            date_cols=TB_DATE_COLS,
            batch_size=128,
            max_epochs=30,
            acc="gpu",
            n_trials=N_TRIALS,
            random_state=RANDOM_STATE,
            checkpoint_dir=model_path / "checkpoint",
            save_dir=model_path / "best_model"
        )
        all_gandalf[tgt_name].fit(tb_no_std_train, tb_no_std_valid, target_cols=tgts)
else:
    for tgt_name, model_path in gandalf_model_load_paths.items():
        all_gandalf[tgt_name] = GandalfRegressor.load_model(save_dir=model_path)


  0%|          | 0/1500 [00:00<?, ?it/s]

Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set t

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │     2.320364236831665     │
│        test_loss_0        │     0.803600549697876     │
│        test_loss_1        │    0.8844174742698669     │
│        test_loss_2        │    0.6323463320732117     │
│  test_mean_squared_error  │     2.320364236831665     │
│ test_mean_squared_error_0 │     0.803600549697876     │
│ test_mean_squared_error_1 │    0.8844174742698669     │
│ test_mean_squared_error_2 │    0.6323463320732117     │
└───────────────────────────┴───────────────────────────┘

In [12]:
gandalf_eval_path = RESULT_DIR / f"gandalf_eval/gandalf_eval_{run_date}"
gandalf_color_eval_path = gandalf_eval_path / "color_model"
gandalf_transmittance_eval_path = gandalf_eval_path / "transmittance_model"

In [13]:
from src.evaluation.evaluator import RegressionEvaluator
from src.constants import *
all_gandalf_color = all_gandalf["color"]
all_gandalf_transmittance = all_gandalf["transmittance"]
all_gandalf_list = [all_gandalf_color, all_gandalf_transmittance]
all_gandalf_eval_paths = [
    gandalf_color_eval_path, 
    gandalf_transmittance_eval_path
]
for model, path, tgts in zip(all_gandalf_list, all_gandalf_eval_paths, TB_TARGETS_LIST):
    gandalf_evaluator = RegressionEvaluator(
        num_cols=TB_NUM_COLS,
        cat_cols=TB_CAT_COLS,
        output_dir=path,
        date_cols=RAW_DATE_COLS,
    )
    gandalf_evaluator.run_full_evaluation(
        regressors=model,
        train=tb_no_std_train,
        valid=tb_no_std_valid,
        target_cols=tgts
    )

  0%|          | 0/1536 [00:00<?, ?it/s]

  0%|          | 0/1536 [00:00<?, ?it/s]

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.metrics import root_mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

from tabpfn import TabPFNRegressor

results = []

for tgt in COLOR_COLS + TRANSMITTANCE_COLS:
    X = tb_data_df[
        TB_FEATURE_COLS 
    ].copy()
    y = tb_data_df[tgt].copy()
    (
        X_train, X_test, 
        y_train, y_test
    ) = train_test_split(
        X, 
        y, 
        test_size=0.2, 
        random_state=RANDOM_STATE
    )
    regressor = TabPFNRegressor(
        categorical_features_indices=[0],
        device="cuda:1",
        model_path="/home_ext/zzx/work_file/other/model_lab/tabpfn/tabpfn-v2.5-regressor-v2.5_default.ckpt",
        random_state=RANDOM_STATE,1. 
    )
    regressor.fit(X_train, y_train)
    train_pred = regressor.predict(X_train)
    valid_pred = regressor.predict(X_test)
    train_r2 = r2_score(y_train, train_pred)
    valid_r2 = r2_score(y_test, valid_pred)
    valid_rmse = root_mean_squared_error(y_test, valid_pred)
    results.append({
        "Target": tgt,
        "Train_R2": round(train_r2, 4),
        "Valid_R2": round(valid_r2, 4),
        "Valid_RMSE": round(valid_rmse, 4),
    })

result_df = pd.DataFrame(results)

print(result_df)


                     Target  Train_R2  Valid_R2  Valid_RMSE
0               C3_10_Y_min    0.3581    0.1977      0.1415
1               C3_10_Y_max    0.6829    0.6219      0.0775
2               C3_10_a_min    0.6338    0.5650      0.1199
3               C3_10_a_max    0.5155    0.4683      0.1668
4               C3_10_b_min    0.7097    0.6855      0.3051
5               C3_10_b_max    0.8258    0.7700      0.1864
6               C3_45_a_min    0.7120    0.7961      0.0896
7               C3_45_a_max    0.8803    0.8579      0.0845
8               C3_45_b_min    0.6164    0.6344      0.1651
9               C3_45_b_max    0.7942    0.7805      0.1203
10   T_0Deg_400_770_AVG_min    0.0242    0.0195      0.8904
11           T_0Deg_940_min    0.0904    0.0534      0.9312
12  T_40Deg_920_960_AVG_min    0.1393    0.2137      0.7504


In [ ]:


# Initialize the regressor

# Uses TabPFN-2.5 weights, trained on synthetic data only.
# To use TabPFN v2:
# regressor = TabPFNRegressor.create_default_for_version(ModelVersion.V2)

predictions = regressor.predict(X_test)
mse = mean_squared_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print("Mean Squared Error (MSE):", mse)
print("R² Score:", r2)

Mean Squared Error (MSE): 0.5631654882668711
R² Score: 0.21374257396696705


# 功能测试

In [1]:
import numpy as np
import pandas as pd
np.random.seed(42)
n = 3000
# 数值特征
x_pos = np.random.normal(0, 1, n)
x_neg = np.random.normal(0, 1, n)
x_nonlin = np.random.normal(0, 1, n)
x_noise = np.random.normal(0, 1, n)
# 分类特征
cat_feat = np.random.choice(["A", "B", "C"], size=n)
# 分类对 y1 的影响
cat_effect_y1 = {
    "A": 5.0,   # 强正
    "B": -5.0,  # 强负
    "C": 0.0    # 无影响
}
# 分类对 y2 的影响
cat_effect_y2 = {
    "A": -3.0,
    "B": 3.0,
    "C": 0.0
}
cat_contrib_y1 = np.array([cat_effect_y1[c] for c in cat_feat])
cat_contrib_y2 = np.array([cat_effect_y2[c] for c in cat_feat])
noise = np.random.normal(0, 0.5, n)
# 构造目标
y1 = (
    3*x_pos
    - 2*x_neg
    + 0.5*(x_nonlin**2)
    + cat_contrib_y1
    + noise
)
y2 = (
    -4*x_pos
    + 1.5*x_neg
    + cat_contrib_y2
    + noise
)
df = pd.DataFrame({
    "x_pos": x_pos,
    "x_neg": x_neg,
    "x_nonlin": x_nonlin,
    "x_noise": x_noise,
    "cat_feat": cat_feat,
    "y1": y1,
    "y2": y2,
})
# 记录变量类型
num_cols = ["x_pos", "x_neg", "x_nonlin", "x_noise"]
cat_cols = ["cat_feat"]
target_cols = ["y1", "y2"]

df.head()

,x_pos,x_neg,x_nonlin,x_noise,cat_feat,y1,y2
0,0.496714,-1.907808,-1.114081,0.765402,B,0.651002,-2.123912
1,-0.138264,-0.860385,-0.630931,1.073413,A,6.642296,-3.600239
2,0.647689,-0.413606,-0.942060,0.498690,A,8.390159,-6.035019
3,1.523030,1.887688,-0.547996,-1.942498,B,-4.354931,-0.559383
4,-0.234153,0.556553,-0.214150,-0.155422,C,-1.195495,2.368584


In [2]:
from sklearn.model_selection import train_test_split
train, valid = train_test_split(df, test_size=0.2, random_state=42)
print(len(train))
print(len(valid))

2400
600


# 单变量模型测试

In [34]:
from src.models.xgb_regressor import XGBRegressor
from src.evaluation.evaluator import RegressionEvaluator

xgb_models = {}
for tgt in target_cols:
    model = XGBRegressor(
        num_cols=num_cols,
        cat_cols=cat_cols,
        num_boost_round=500,
        early_stopping_rounds=30,
        n_trials=10,
        n_jobs=1,
        random_state=42
    )
    model.fit(
        train=train,
        valid=valid,
        target_cols=[tgt]
    )
    xgb_models[tgt] = model
evaluator = RegressionEvaluator(
    num_cols=num_cols,
    cat_cols=cat_cols,
    output_dir="test/xgb_test/eval"
)
evaluator.run_full_evaluation(
    regressors=xgb_models,
    train=train,
    valid=valid,
    target_cols=target_cols
)

  0%|          | 0/10 [00:00<?, ?it/s]

[0]	valid-rmse:5.45402
[100]	valid-rmse:0.99063
[200]	valid-rmse:0.71076
[300]	valid-rmse:0.68367
[400]	valid-rmse:0.68088
[499]	valid-rmse:0.67902


  0%|          | 0/10 [00:00<?, ?it/s]

[0]	valid-rmse:4.93045
[100]	valid-rmse:0.93105
[200]	valid-rmse:0.70866
[300]	valid-rmse:0.69238
[400]	valid-rmse:0.68925
[499]	valid-rmse:0.68783


  0%|          | 0/600 [00:00<?, ?it/s]

  0%|          | 0/600 [00:00<?, ?it/s]

In [7]:
from src.models.gandalf_regressor import GandalfRegressor
from src.evaluation.evaluator import RegressionEvaluator

gandalf_model = GandalfRegressor(
    continuous_cols=num_cols,
    categorical_cols=cat_cols,
    date_cols=[],
    batch_size=64,
    max_epochs=10,
    n_trials=10,
    n_jobs=1,
    acc="cpu",
    num_workers=10
)
gandalf_model.fit(
    train=train,
    valid=valid,
    target_cols=target_cols,
)
evaluator = RegressionEvaluator(
    num_cols=num_cols,
    cat_cols=cat_cols,
    output_dir="test/gandalf_test/eval"
)
evaluator.run_full_evaluation(
    regressors=gandalf_model,
    train=train,
    valid=valid,
    target_cols=target_cols
)

  0%|          | 0/10 [00:00<?, ?it/s]

Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42
Seed set to 42


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │    2.4437928199768066     │
│        test_loss_0        │    1.5371805429458618     │
│        test_loss_1        │    0.9066124558448792     │
│  test_mean_squared_error  │    2.4437928199768066     │
│ test_mean_squared_error_0 │    1.5371805429458618     │
│ test_mean_squared_error_1 │    0.9066124558448792     │
└───────────────────────────┴───────────────────────────┘

  0%|          | 0/600 [00:00<?, ?it/s]